# L0–L2 · 数据层、边缘分布、尾部相关性

本 notebook 覆盖三层，每一步都先落成 DataFrame 再往下算：

| 层 | 内容 | 产出 |
|---|---|---|
| L0 | 数据提取与质量体检 | 价格表、收益率表、质量表 |
| L1 | 边缘分布与概率积分变换（PIT） | PIT 表（三种口径并排） |
| L2 | Copula 与滚动尾部相关系数 | λ_D / λ_U 序列 |

**核心数学在 `tailrisk_core.py`**（Copula 的参数边界和独立性检验踩过坑，
有回归测试背书，不在 notebook 里重写）。数据提取、展示、图表都在这里。

## 0 · 环境

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tailrisk_core as tc

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.unicode_minus"] = False
# 中文字体：Mac 用 Arial Unicode MS，Windows 用 SimHei
for f in ["Arial Unicode MS", "SimHei", "PingFang SC"]:
    try:
        plt.rcParams["font.sans-serif"] = [f]
        break
    except Exception:
        pass

print("numpy", np.__version__, "| pandas", pd.__version__)

---
# L0 · 数据提取

两个来源：**聚宽**（境内指数，历史到 2005）和**同花顺 Excel 导出**（中债综合等）。

先各自取出来看 DataFrame，确认没问题再合并。

### L0.1 聚宽

In [ ]:
import os
import jqdatasdk as jq


def jq_login():
    """
    聚宽登录，三级回退：环境变量 → credentials.py → 交互输入。

    为什么不直接用 os.environ：终端里 export 的变量只有从该终端启动的进程
    才继承得到。从 Dock/Spotlight 点开的 VS Code 拿不到，其内核也拿不到。
    Jupyter 若从终端启动则没问题。三级回退让哪种启动方式都能跑。

    ⚠ credentials.py 记得加进 .gitignore，别让密码进版本库或跟着压缩包发出去。
    """
    u, p = os.environ.get("JQ_USERNAME"), os.environ.get("JQ_PASSWORD")
    if u and p:
        src = "环境变量"
    else:
        try:
            import credentials
            u, p = credentials.JQ_USERNAME, credentials.JQ_PASSWORD
            src = "credentials.py"
        except (ImportError, AttributeError):
            from getpass import getpass
            u, p = input("聚宽账号: "), getpass("密码: ")
            src = "交互输入"
    jq.auth(u, p)
    print(f"聚宽登录成功（凭据来自{src}）")
    return jq


jq_login()

START, END = "2005-01-01", "2026-08-17"

JQ_ASSETS = {
    "沪深300":      "000300.XSHG",
    "中证500":      "000905.XSHG",
    "上证国债指数":  "000012.XSHG",
    "上证企业债指数": "000013.XSHG",
}

jq_px = {}
for name, code_ in JQ_ASSETS.items():
    s = jq.get_price(code_, start_date=START, end_date=END, fields=["close"])["close"]
    jq_px[name] = s.dropna()

jq_px = pd.DataFrame(jq_px)
jq_px.head()

In [ ]:
# 每个标的的实际覆盖区间
pd.DataFrame({
    "起点": jq_px.apply(lambda s: s.first_valid_index()),
    "终点": jq_px.apply(lambda s: s.last_valid_index()),
    "非空观测": jq_px.notna().sum(),
})

### L0.2 同花顺 Excel 导出

同花顺导出是**多标的横向拼接**格式，每个标的占 3 列（日期 / 数值 / 空列），
且**各标的日期列相互独立**——不同市场交易日不同，不能按行号对齐拼接。
`parse_ifind_export` 逐标的取 (日期, 数值) 配对后再按日期外连接。

导出设置务必确认：日期类型=**交易日**（不填充非交易日）、
价格类型(债券)=**全价/财富**（不是净价）、时间排序=升序。

In [ ]:
THS_PATH = "data/ifind_export.txt"   # 制表符分隔，改成你的实际路径

import pathlib
if pathlib.Path(THS_PATH).exists():
    ths_px = tc.parse_ifind_export(THS_PATH)
    print(f"解析出 {ths_px.shape[1]} 个标的，{ths_px.shape[0]} 行")
    display(ths_px.head())
else:
    print(f"未找到 {THS_PATH}，跳过同花顺部分")
    ths_px = pd.DataFrame()

In [ ]:
if not ths_px.empty:
    display(pd.DataFrame({
        "起点": ths_px.apply(lambda s: s.first_valid_index()),
        "终点": ths_px.apply(lambda s: s.last_valid_index()),
        "非空观测": ths_px.notna().sum(),
    }))

### L0.3 合并

外连接保留所有交易日。**不做前向填充**——填充会产生大量重复值，
后面 PIT 时这些 ties 会挤在同一分位，让 Copula 的 MLE 偏向独立解。

In [ ]:
prices = pd.concat([jq_px, ths_px], axis=1).sort_index() if not ths_px.empty else jq_px.copy()
print(f"合并后 {prices.shape[0]} 行 × {prices.shape[1]} 列")
prices.tail()

### L0.4 数据质量体检

In [ ]:
quality = tc.data_quality(prices)
quality

**看什么**：

- `零收益占比` > 5% 或 `唯一值占比` < 80% → ties 问题，Copula 会偏向独立解
- `年化收益` 接近票息水平（3–4%）且回撤很小 → 财富指数；接近 0 且震荡 → 净价指数
- `起点` 决定能覆盖哪几次历史危机

### L0.5 收益率

In [ ]:
returns = prices.pct_change().dropna(how="all")
returns.describe().T[["count", "mean", "std", "min", "max"]]

---
# L1 · 边缘分布与 PIT

Copula 的输入必须是 (0,1) 上的均匀分布值，需要先把收益率做概率积分变换。

**三种口径并排**，因为这一步的选择会实质影响结论：

| 口径 | 做法 | 定位 |
|---|---|---|
| `empirical` | 直接对收益率做秩变换 | **最贴近中金原文**（原文未提任何滤波） |
| `garch` | GJR-GARCH 滤波后对标准化残差做秩变换 | 本项目对照版本 |
| `vol_scaled` | 滚动波动率标准化后秩变换 | 中间档 |

⚠ GARCH 滤波会剥离波动率联动，而危机期「股债同步暴跌」很大程度就表现为
两者波动率同时放大——可能把要测的东西一起剥掉。模拟检验：残差独立、
仅波动率由共同因子驱动的数据，滤波前 λ_D>0 占 6.4%，滤波后掉到 2.2%（噪声基准）。

In [ ]:
# 选两个资产做演示，后面中金复现用的就是这一对
A, B = "沪深300", "上证国债指数"

pit_table = pd.DataFrame({
    f"{A}_empirical": tc.pit_empirical(returns[A]),
    f"{B}_empirical": tc.pit_empirical(returns[B]),
    f"{A}_vol_scaled": tc.pit_vol_scaled(returns[A]),
    f"{B}_vol_scaled": tc.pit_vol_scaled(returns[B]),
})
pit_table.head()

In [ ]:
# GARCH 较慢，单独一格
pit_table[f"{A}_garch"] = tc.pit_garch(returns[A])
pit_table[f"{B}_garch"] = tc.pit_garch(returns[B])
pit_table.describe().T[["count", "mean", "std", "min", "max"]]

PIT 后的值应该近似均匀分布：均值≈0.5、标准差≈0.289、范围在 (0,1) 内。
偏离说明边缘分布没处理干净。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, kind in zip(axes, ["empirical", "vol_scaled", "garch"]):
    ax.hist(pit_table[f"{A}_{kind}"].dropna(), bins=40, alpha=.7)
    ax.axhline(len(pit_table[f"{A}_{kind}"].dropna())/40, color="r", ls="--", lw=1)
    ax.set_title(f"{A} · {kind}")
plt.suptitle("PIT 后应接近均匀分布（红线=理论水平）")
plt.tight_layout(); plt.show()

---
# L2 · Copula 与尾部相关性

## L2.1 单窗口示例：四种 Copula 对比

先在全样本上拟合一次，看各族的 λ 取值差异。

In [ ]:
u = pit_table[f"{A}_empirical"].dropna()
v = pit_table[f"{B}_empirical"].dropna()
idx = u.index.intersection(v.index)
u, v = u.reindex(idx).values, v.reindex(idx).values

rows = []
cl = tc.ClaytonCopula().fit(u, v)
gu = tc.GumbelCopula().fit(u, v)
ga = tc.GaussianCopula().fit(u, v)
for m, ld, lu in [(cl, cl.lambda_lower, 0.0), (gu, 0.0, gu.lambda_upper), (ga, 0.0, 0.0)]:
    rows.append({"copula": m.name, "theta/rho": getattr(m, "theta", getattr(m, "rho", np.nan)),
                 "loglik": m.loglik, "lambda_D": ld, "lambda_U": lu,
                 "独立性p值": getattr(m, "indep_pvalue", np.nan)})
pd.DataFrame(rows)

## L2.2 噪声基准

**这是全项目最重要的一张表。** 任何「测出了尾部相关性」的结论，
都必须和「纯独立数据下同一统计量的取值」对照，否则无法区分信号与噪声。

首轮中金复现之所以能定性为「信号未产生」而非「信号很弱」，靠的就是这个对照。

In [ ]:
baselines = pd.DataFrame([
    tc.noise_baseline(window=60, n_rep=300, rho=0.0, seed=1),
    tc.noise_baseline(window=60, n_rep=300, rho=0.3, seed=2),
    tc.noise_baseline(window=60, n_rep=300, rho=0.5, seed=3),
])
baselines

**判读**：

- `rho=0` 那行的 `lambda_D>0占比` 就是**误报率下限**（约 2%）。实测值若与之接近，
  说明测出的「下尾相关」与噪声不可区分。
- 随 rho 上升，`lambda_U>0占比` 急剧上升 → 中金的「λ_U=0」条件在相关性较高时
  可能**永不成立**。这是已知的方法局限，见 `tailrisk_core.GumbelCopula` 的注释。

## L2.3 滚动尾部相关系数

In [ ]:
roll = tc.rolling_tail_dependence(
    pit_table[f"{A}_empirical"], pit_table[f"{B}_empirical"],
    window=60, step=1)
print(f"{len(roll)} 个重估点")
roll.head()

In [ ]:
summary = pd.DataFrame({
    "lambda_D": [(roll.lambda_D > tc.LAMBDA_ZERO_TOL).mean(),
                 roll.lambda_D[roll.lambda_D > tc.LAMBDA_ZERO_TOL].median()],
    "lambda_U": [(roll.lambda_U > tc.LAMBDA_ZERO_TOL).mean(),
                 roll.lambda_U[roll.lambda_U > tc.LAMBDA_ZERO_TOL].median()],
}, index=["大于0的占比", "非零时的中位数"])
summary

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
axes[0].plot(roll.index, roll.lambda_D, lw=.8, color="firebrick")
axes[0].set_ylabel("λ_D 下尾"); axes[0].set_title("滚动尾部相关系数（60日窗口）")
axes[1].plot(roll.index, roll.lambda_U, lw=.8, color="steelblue")
axes[1].set_ylabel("λ_U 上尾")
for ax in axes: ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## L2.4 尾部不对称诊断

`ll_diff = ll_Clayton − ll_Gumbel` 是对「λ_U=0」条件的补充判据。

模拟参照：真下尾数据均值 **+12**（99.7% 为正），真上尾数据均值 **−16**（0% 为正），
对称/无尾部相关数据在 **0** 附近。

In [ ]:
dll = roll.ll_diff.dropna()
diag = pd.DataFrame({
    "似然差分位数": dll.quantile([.01, .25, .5, .75, .95, .99]),
})
display(diag)

ct = pd.crosstab(
    (roll.lambda_D > tc.LAMBDA_ZERO_TOL).map({True: "λ_D>0", False: "λ_D=0"}),
    (roll.lambda_U > tc.LAMBDA_ZERO_TOL).map({True: "λ_U>0", False: "λ_U=0"}),
)
print("\nλ_D × λ_U 联合分布（天数）:")
display(ct)
print("占比 (%):")
display((ct / ct.values.sum() * 100).round(1))

---
## 小结

到这里 L0–L2 的机器已经就位：数据、边缘分布、滚动尾部相关系数。

下一步进 `02_中金复现.ipynb`，在这套机器上加**趋势条件**和**预警状态机**，
复现中金原文的股债双杀预警。